Import all necessary libraries.

In [1]:
import pandas as pd
import numpy as np
import os
import wandb
import random
import math
import torch
import torch.nn as nn
from torch.nn.utils import clip_grad_norm_
from torch.utils.data import DataLoader
from imblearn.metrics import geometric_mean_score
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import f1_score
import matplotlib.pyplot as plt
from types import SimpleNamespace

from xgboost import XGBClassifier

from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import cross_val_score

from sklearn.metrics import confusion_matrix
import seaborn as sns

import sys
sys.path.append(os.path.join(os.getcwd(), '../src'))

from transforms.feature_engineering_classification import add_all_features, filter_business_hours, entries_per_day_per_site
from transforms.feature_engineering_classification import (
    CONTINUOUS_FEATURE_COLUMNS,
    CATEGORICAL_FEATURE_COLUMNS,
    CYCLIC_FEATURE_COLUMNS,
    TARGET_COLUMN
)
from evaluation.comp_metrics import evaluate_all_metrics
from evaluation.visual import plot_confusion_matrix

from datasets.flextrack_dataset import FlextrackClassificationDataset
from utils.losses import FocalLoss

SWEEP = True

c:\Users\timon\.pyenv-win-venv\envs\aicomp\Lib\site-packages\pydantic\_internal\_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
c:\Users\timon\.pyenv-win-venv\envs\aicomp\Lib\site-packages\pydantic\_internal\_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'frozen' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'frozen' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because a

Set seed for reproducibility.

In [2]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

In [3]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cuda


In [4]:
os.environ["WANDB_API_KEY"] = "3aaf9f796df65417b3f5f8560b43875171b55805"

wandb.login()

wandb: Currently logged in as: fabian-dubach (fabian-dubach-hochschule-luzern) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [5]:
# just use regression data and remove DR-Flags and DR-Capacity
df_train = pd.read_csv(os.path.normpath(os.path.join(os.getcwd(), '../data/regression/regression-train.csv')))
df_test = pd.read_csv(os.path.normpath(os.path.join(os.getcwd(), '../data/regression/regression-test.csv')))

In [6]:
print(f"Dataset shape: {df_train.shape}")
print(f"\nColumns: {df_train.columns.tolist()}")

Dataset shape: (105120, 7)

Columns: ['Site', 'Timestamp_Local', 'Dry_Bulb_Temperature_C', 'Global_Horizontal_Radiation_W/m2', 'Building_Power_kW', 'Demand_Response_Flag', 'Demand_Response_Capacity_kW']


# Feature Engineering

We use same feature engineering as in regression task but remove the irrelevant features.

In [7]:
df_train = add_all_features(df_train)
# TODO: add lag features for non sequence training (like for trees or tcn etc...)

df_train = filter_business_hours(df_train)

ENTRIES_PER_DAY = entries_per_day_per_site(df_train)

In [8]:
print(f"Dataset shape: {df_train.shape}")
print(f"\nColumns: {df_train.columns.tolist()}")

Dataset shape: (58035, 40)

Columns: ['Site', 'Timestamp_Local', 'Dry_Bulb_Temperature_C', 'Global_Horizontal_Radiation_W/m2', 'Building_Power_kW', 'Demand_Response_Flag', 'Demand_Response_Capacity_kW', 'hour', 'minute', 'day_of_week', 'is_weekend', 'is_holiday', 'month_sin', 'month_cos', 'Building_Power_kW_diff_15min', 'Building_Power_kW_diff_1h', 'Building_Power_kW_diff_1d', 'Dry_Bulb_Temperature_C_diff_15min', 'Global_Horizontal_Radiation_W/m2_diff_15min', 'Building_Power_kW_rolling_mean_1h', 'Building_Power_kW_rolling_mean_2h', 'Building_Power_kW_rolling_mean_1d', 'Building_Power_kW_rolling_std_1h', 'Building_Power_kW_rolling_std_2h', 'Building_Power_kW_rolling_std_1d', 'Building_Power_kW_rolling_min_1h', 'Building_Power_kW_rolling_min_2h', 'Building_Power_kW_rolling_max_1h', 'Building_Power_kW_rolling_max_2h', 'minute_0', 'minute_15', 'minute_30', 'minute_45', 'day_of_week_0', 'day_of_week_1', 'day_of_week_2', 'day_of_week_3', 'day_of_week_4', 'day_of_week_5', 'day_of_week_6']


# Split sites

In [9]:
def count_sites(df):

    counter_site_a = 0
    counter_site_b = 0
    counter_site_c = 0
    counter_site_d = 0
    counter_site_e = 0

    for i in df['Site']:
        if i == 'siteA':
            counter_site_a += 1
        elif i == 'siteB':
            counter_site_b += 1
        elif i == 'siteC':
            counter_site_c += 1
        elif i == 'siteD':
            counter_site_d += 1
        elif i == 'siteE':
            counter_site_e += 1
    
    return counter_site_a, counter_site_b, counter_site_c, counter_site_d, counter_site_e

In [10]:
df_train_site_a = df_train[0:19345]
count_sites(df_train_site_a)

df_train_site_b = df_train[19345:38690]
count_sites(df_train_site_b)

df_train_site_c = df_train[38690:58035]
count_sites(df_train_site_c)

(0, 0, 19345, 0, 0)

In [11]:
X_continuous_site_a = df_train_site_a[CONTINUOUS_FEATURE_COLUMNS].values # Convert to numpy array
X_continuous_site_b = df_train_site_b[CONTINUOUS_FEATURE_COLUMNS].values # Convert to numpy array
X_continuous_site_c = df_train_site_c[CONTINUOUS_FEATURE_COLUMNS].values # Convert to numpy array

X_categorical_site_a = df_train_site_a[CATEGORICAL_FEATURE_COLUMNS].values # Convert to numpy array
X_categorical_site_b = df_train_site_b[CATEGORICAL_FEATURE_COLUMNS].values # Convert to numpy array
X_categorical_site_c = df_train_site_c[CATEGORICAL_FEATURE_COLUMNS].values # Convert to numpy array

X_cyclic_site_a = df_train_site_a[CYCLIC_FEATURE_COLUMNS].values # Convert to numpy array
X_cyclic_site_b = df_train_site_b[CYCLIC_FEATURE_COLUMNS].values # Convert to numpy array
X_cyclic_site_c = df_train_site_c[CYCLIC_FEATURE_COLUMNS].values # Convert to numpy array

y_site_a = df_train_site_a[TARGET_COLUMN].values
y_site_b = df_train_site_b[TARGET_COLUMN].values
y_site_c = df_train_site_c[TARGET_COLUMN].values

In [12]:
print(f"Continuous feature shape: {X_continuous_site_a.shape}")
print(f"Continuous feature shape: {X_continuous_site_b.shape}")
print(f"Continuous feature shape: {X_continuous_site_c.shape}")

print(f"Categorical feature shape: {X_categorical_site_a.shape}")
print(f"Categorical feature shape: {X_categorical_site_b.shape}")
print(f"Categorical feature shape: {X_categorical_site_c.shape}")

print(f"Cyclic feature shape: {X_cyclic_site_a.shape}")
print(f"Cyclic feature shape: {X_cyclic_site_b.shape}")
print(f"Cyclic feature shape: {X_cyclic_site_c.shape}")

print(f"Target shape: {y_site_a.shape}")
print(f"Target shape: {y_site_b.shape}")
print(f"Target shape: {y_site_c.shape}")

Continuous feature shape: (19345, 19)
Continuous feature shape: (19345, 19)
Continuous feature shape: (19345, 19)
Categorical feature shape: (19345, 13)
Categorical feature shape: (19345, 13)
Categorical feature shape: (19345, 13)
Cyclic feature shape: (19345, 2)
Cyclic feature shape: (19345, 2)
Cyclic feature shape: (19345, 2)
Target shape: (19345,)
Target shape: (19345,)
Target shape: (19345,)


# Normalization

In [13]:
scaler_X_site_a = StandardScaler()
scaler_X_site_b = StandardScaler()
scaler_X_site_c = StandardScaler()

In [14]:
X_scaled_site_a = scaler_X_site_a.fit_transform(X_continuous_site_a)
X_scaled_site_b = scaler_X_site_b.fit_transform(X_continuous_site_b)
X_scaled_site_c = scaler_X_site_c.fit_transform(X_continuous_site_c)

Concatenate the unscaled and the scaled features together.

In [15]:
X_site_a = np.concatenate([X_scaled_site_a, X_categorical_site_a, X_cyclic_site_a], axis=1)
X_site_b = np.concatenate([X_scaled_site_b, X_categorical_site_b, X_cyclic_site_b], axis=1)
X_site_c = np.concatenate([X_scaled_site_c, X_categorical_site_c, X_cyclic_site_c], axis=1)

In [16]:
X_site_a = X_site_a.astype(np.float32)
X_site_b = X_site_b.astype(np.float32)
X_site_c = X_site_c.astype(np.float32)

y_site_a = y_site_a.astype(int)
y_site_b = y_site_b.astype(int)
y_site_c = y_site_c.astype(int)

In [17]:
print(f"All features shape: {X_site_a.shape}")
print(f"Target shape: {y_site_a.shape}")

All features shape: (19345, 34)
Target shape: (19345,)


In [18]:
np.unique(y_site_a)

array([0, 1, 2])

IMPORTANT: Remove entries, where features are incomplete (at start of dataset)

In [19]:
print("First few entries of each site have nan values due to feature engineering:\n", X_site_a[0])
print(X_site_a[ENTRIES_PER_DAY])

First few entries of each site have nan values due to feature engineering:
 [ 0.57910997 -1.3638599  -0.5221737  -0.00325376 -0.00751908         nan
 -0.69390106  0.0624692  -0.54411376 -0.5452835          nan -0.64983785
 -0.84162885         nan -0.38954532 -0.25569385 -0.6574576  -0.75812143
 -1.602483    1.          0.          0.          0.          0.
  1.          0.          0.          0.          0.          0.
  0.          1.          0.5         0.8660254 ]
[ 3.1494236e-01 -1.3638599e+00 -5.2217370e-01 -3.2537556e-03
 -7.5190784e-03  9.0518305e-03 -3.7747535e-01  6.2469199e-02
 -5.4411376e-01 -5.4528350e-01  2.9363585e+00 -6.4983785e-01
 -8.4162885e-01  4.3428288e+00 -3.8954532e-01 -2.5569385e-01
 -6.5745759e-01 -7.5812143e-01 -1.6024830e+00  1.0000000e+00
  0.0000000e+00  0.0000000e+00  0.0000000e+00  0.0000000e+00
  0.0000000e+00  1.0000000e+00  0.0000000e+00  0.0000000e+00
  0.0000000e+00  0.0000000e+00  0.0000000e+00  0.0000000e+00
  5.0000000e-01  8.6602539e-01]


In [20]:
# Create mask to exclude first ENTRIES_PER_DAY of each site
mask_site_a = np.ones(len(X_site_a), dtype=bool)
mask_site_b = np.ones(len(X_site_b), dtype=bool)
mask_site_c = np.ones(len(X_site_c), dtype=bool)

# Site A: exclude indices 0 to ENTRIES_PER_DAY-1
mask_site_a[0:ENTRIES_PER_DAY] = False

# Site B: exclude indices (365*ENTRIES_PER_DAY) to (365*ENTRIES_PER_DAY + ENTRIES_PER_DAY-1)
mask_site_b[0:ENTRIES_PER_DAY] = False

# Site C: exclude indices (730*ENTRIES_PER_DAY) to (730*ENTRIES_PER_DAY + ENTRIES_PER_DAY-1)
mask_site_c[0:ENTRIES_PER_DAY] = False

# Apply mask to remove incomplete entries
X_site_a = X_site_a[mask_site_a]
X_site_b = X_site_b[mask_site_b]
X_site_c = X_site_c[mask_site_c]

y_site_a = y_site_a[mask_site_a]
y_site_b = y_site_b[mask_site_b]
y_site_c = y_site_c[mask_site_c]

### Data Splitting

In [21]:
# Calculate split indices (accounting for removed incomplete entries)
site_a_entries = (365 - 1) * ENTRIES_PER_DAY  # 364 days of site A
site_b_entries = (365 - 1) * ENTRIES_PER_DAY  # 364 days of site B

# Train on Site A and Site C, validate on Site B
# X_train = np.vstack((X_site_a, X_site_c))
# X_val = X_site_b
# y_train = np.vstack((y_site_a, y_site_c))
# y_val = y_site_b

In [22]:
# Keep per-site arrays in one place (for CV)
sites = {
    "SiteA": (X_site_a, y_site_a),
    "SiteB": (X_site_b, y_site_b),
    "SiteC": (X_site_c, y_site_c),
}

# print(len(X_train))
# print(len(y_train))
# print(len(X_val))
# print(len(y_val))

# Parameters

In [23]:
# ============================================================================
# 2. CONFIGURE HYPERPARAMETERS FOR MULTI-CLASS
# ============================================================================

# https://wandb.ai/fabian-dubach-hochschule-luzern/AICOMP_Flextrack/sweeps/c4ctfqbn/runs/81hdc7oe
single_run_config = {
    # training loop
    "num_boost_round": 300,
    "early_stopping_rounds": 50,
    "eta": 0.01555986490319607,

    # tree complexity
    "max_depth": 5,
    "min_child_weight": 5.282700531432371,
    "gamma": 4.618648898031871,

    # sampling
    "subsample": 0.9534271279411476,
    "colsample_bytree": 0.7717558415567605,

    # regularization
    "reg_alpha": 0.00000005337578063702,
    "reg_lambda": 17.97610553993505,
}

single_run_config = SimpleNamespace(**single_run_config)

# Sweep

In [24]:
import wandb

sweep_config = {
    "method": "bayes",
    "metric": {
        "name": "val/f1",
        "goal": "maximize",
    },
    "parameters": {
        # training loop
        "num_boost_round": {
            "values": [300, 500, 800, 1200]
        },
        "early_stopping_rounds": {
            "values": [30, 50, 80]
        },

        # learning rate
        "eta": {
            "distribution": "log_uniform_values",
            "min": 0.005,
            "max": 0.2,
        },

        # tree complexity
        "max_depth": {
            "values": [3, 4, 5, 6, 7, 8, 10]
        },
        "min_child_weight": {
            "distribution": "log_uniform_values",
            "min": 1,
            "max": 20,
        },
        "gamma": {
            "distribution": "uniform",
            "min": 0.0,
            "max": 5.0,
        },

        # sampling
        "subsample": {
            "distribution": "uniform",
            "min": 0.6,
            "max": 1.0,
        },
        "colsample_bytree": {
            "distribution": "uniform",
            "min": 0.6,
            "max": 1.0,
        },

        # regularization
        "reg_alpha": {
            "distribution": "log_uniform_values",
            "min": 1e-8,
            "max": 1.0,
        },
        "reg_lambda": {
            "distribution": "log_uniform_values",
            "min": 0.5,
            "max": 20.0,
        },
    },
}


In [25]:
import numpy as np
import xgboost as xgb
from collections import Counter

from imblearn.over_sampling import SMOTE
from wandb.integration.xgboost import WandbCallback

from sklearn.metrics import accuracy_score, confusion_matrix, f1_score
from imblearn.metrics import geometric_mean_score


BASE_PARAMS = {
    "objective": "multi:softprob",
    "num_class": 3,
    "eval_metric": ["mlogloss", "merror"],
    "tree_method": "hist",
    "random_state": 42,
    "n_jobs": -1,
}

CLASS_NAMES = ["Decrease (-1)", "No Change (0)", "Increase (+1)"]


def train(config=None, X_train=None, y_train=None, X_val=None, y_val=None, fold_name=None):
    
    if config is None:
        run = wandb.init(
            project="AICOMP_Flextrack",
            entity="fabian-dubach-hochschule-luzern",
        )

        cfg = wandb.config
    else:
        run = wandb.init(
            # Set the wandb entity where your project will be logged (generally your team name).
            entity="fabian-dubach-hochschule-luzern",
            # Set the wandb project where this run will be logged.
            project="AICOMP_Flextrack",
            # Name this run
            name=f"xgboost-classification-{fold_name}-numBoostRound_{config.num_boost_round}-maxDepth_{config.max_depth}",
            # Track hyperparameters and run metadata.
            config=config
        )

        cfg = wandb.config

    # ---------------------------
    # Data prep
    # ---------------------------
    smote = SMOTE(random_state=42)
    X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)

    dtrain = xgb.DMatrix(X_train_resampled, label=y_train_resampled)
    dval   = xgb.DMatrix(X_val, label=y_val)

    # ---------------------------
    # XGBoost params from sweep
    # ---------------------------
    xgb_params = dict(BASE_PARAMS)
    xgb_params.update({
        "eta": cfg.eta,
        "max_depth": cfg.max_depth,
        "min_child_weight": cfg.min_child_weight,
        "gamma": cfg.gamma,
        "subsample": cfg.subsample,
        "colsample_bytree": cfg.colsample_bytree,
        "reg_alpha": cfg.reg_alpha,
        "reg_lambda": cfg.reg_lambda,
    })

    evals = [(dtrain, "train"), (dval, "val")]
    evals_result = {}

    model = xgb.train(
        params=xgb_params,
        dtrain=dtrain,
        num_boost_round=int(cfg.num_boost_round),
        evals=evals,
        early_stopping_rounds=int(cfg.early_stopping_rounds),
        verbose_eval=False,
        evals_result=evals_result,
        callbacks=[WandbCallback(log_model=False)],
    )

    # ---------------------------
    # Log loss curves
    # ---------------------------
    for step, (tl, vl) in enumerate(
        zip(
            evals_result["train"]["mlogloss"],
            evals_result["val"]["mlogloss"],
        )
    ):
        wandb.log({"train/loss": tl, "val/loss": vl}, step=step)

    # ---------------------------
    # Predictions
    # ---------------------------
    y_train_pred = np.argmax(model.predict(dtrain), axis=1)
    y_val_pred = np.argmax(model.predict(dval), axis=1)

    # ---------------------------
    # Metrics
    # ---------------------------
    metrics = {
        "fold": fold_name,
        "best_iteration": model.best_iteration,
        "train/f1": f1_score(y_train_resampled, y_train_pred, average="macro"),
        "val/f1": f1_score(y_val, y_val_pred, average="macro"),
        "train/geometric_mean": geometric_mean_score(
            y_train_resampled, y_train_pred, average="macro"
        ),
        "val/geometric_mean": geometric_mean_score(
            y_val, y_val_pred, average="macro"
        ),
        "val_predictions": y_val_pred
    }

    wandb.log({
        "best_iteration": metrics["best_iteration"],
        "train/f1": metrics["train/f1"],
        "val/f1": metrics["val/f1"],
        "train/geometric_mean": metrics["train/geometric_mean"],
        "val/geometric_mean": metrics["val/geometric_mean"],
        "train/accuracy": accuracy_score(y_train_resampled, y_train_pred),
        "val/accuracy": accuracy_score(y_val, y_val_pred),
    })

    # Confusion matrices
    wandb.log({
        "train/confusion_matrix": wandb.plot.confusion_matrix(
            y_true=y_train_resampled,
            preds=y_train_pred,
            class_names=CLASS_NAMES,
        ),
        "val/confusion_matrix": wandb.plot.confusion_matrix(
            y_true=y_val,
            preds=y_val_pred,
            class_names=CLASS_NAMES,
        ),
    })

    if config is None:
        run.finish()
    else:
        run.finish()
        # wandb.finish()

    return metrics

In [26]:
SWEEP = False

In [27]:
if SWEEP:
    # Train on Site A and Site C, validate on Site B
    # X_train = np.vstack((X_site_a, X_site_c))
    # X_val = X_site_b
    # y_train = np.vstack((y_site_a, y_site_c))
    # y_val = y_site_b

    sweep_id = wandb.sweep(
        sweep=sweep_config,
        project="AICOMP_Flextrack",
        entity="fabian-dubach-hochschule-luzern",
    )

    print("Sweep ID:", sweep_id)

    wandb.agent(
        sweep_id,
        function=train,
        count=20,  # number of sweep runs
    )

else:
    import numpy as np

    site_names = list(sites.keys())

    # leave-one-site-out CV folds
    folds = []
    for val_site in site_names:
        train_sites = [s for s in site_names if s != val_site]
        folds.append((train_sites, val_site))

    cv_results = []

    for fold_idx, (train_sites, val_site) in enumerate(folds, start=1):
        X_train = np.vstack([sites[s][0] for s in train_sites])
        y_train = np.concatenate([sites[s][1] for s in train_sites])

        X_val, y_val = sites[val_site]

        fold_name = f"fold_{fold_idx}-val_{val_site}"

        # run one fold
        metrics = train(
            config=single_run_config,
            X_train=X_train, y_train=y_train,
            X_val=X_val, y_val=y_val,
            fold_name=fold_name
        )

        cv_results.append(metrics)

    # ------------------------------------------------------------
    # Build an out-of-fold CSV where Demand_Response_Flag is replaced
    # by the predicted flag (OOF predictions per validation fold).
    # ------------------------------------------------------------

    # 1) Collect OOF predictions per validation site (fold)
    # fold_name looks like: "fold_1-val_SiteA"
    oof_pred_by_sitekey = {}
    for r in cv_results:
        # robust parse
        # expected suffix "...-val_<SiteA|SiteB|SiteC>"
        val_site_key = r["fold"].split("-val_")[-1]
        oof_pred_by_sitekey[val_site_key] = np.asarray(r["val_predictions"])

    # 2) Re-load the raw training CSV (7 columns)
    raw_path = os.path.normpath(os.path.join(os.getcwd(), "../data/regression/regression-train.csv"))
    df_raw = pd.read_csv(raw_path)

    RAW_COLS = [
        "Site",
        "Timestamp_Local",
        "Dry_Bulb_Temperature_C",
        "Global_Horizontal_Radiation_W/m2",
        "Building_Power_kW",
        "Demand_Response_Flag",
        "Demand_Response_Capacity_kW",
    ]

    # hard fail if we didn't load the raw file
    extra_cols = [c for c in df_raw.columns if c not in RAW_COLS]
    if extra_cols:
        raise ValueError(f"Input file is not raw. Extra columns found: {extra_cols}")

    # 3) Feature engineering ONLY on a copy (never mutate df_raw)
    df_fe = add_all_features(df_raw.copy())
    df_fe = filter_business_hours(df_fe)

    ENTRIES_PER_DAY_LOCAL = entries_per_day_per_site(df_fe)

    # Keep stable mapping back to df_raw row numbers
    df_fe = df_fe.reset_index(drop=False).rename(columns={"index": "raw_row_id"})

    df_fe_valid = (
        df_fe.sort_values(["Site", "Timestamp_Local"])
            .groupby("Site", group_keys=False)
            .apply(lambda g: g.iloc[ENTRIES_PER_DAY_LOCAL:])
            .reset_index(drop=True)
    )

    # 5) Assign predictions per site in the SAME order as df_fe_valid
    # Your CV uses keys "SiteA", "SiteB", "SiteC" but the column values are 'siteA', ...
    sitekey_to_sitevalue = {"SiteA": "siteA", "SiteB": "siteB", "SiteC": "siteC"}

    # Inverse mapping back to original flags (-1,0,1) as per CLASSIFICATION_MAPPING in feature_engineering_classification.py
    # CLASSIFICATION_MAPPING = {-1:0, 0:1, 1:2}  => inverse is {0:-1, 1:0, 2:1}
    inv_map = {0: -1, 1: 0, 2: 1}

    pred_flag_col = np.full(len(df_fe_valid), np.nan)

    for site_key, site_value in sitekey_to_sitevalue.items():
        mask = (df_fe_valid["Site"] == site_value).values
        preds = oof_pred_by_sitekey[site_key]

        if preds.shape[0] != mask.sum():
            raise ValueError(
                f"Prediction length mismatch for {site_key}/{site_value}: "
                f"preds={preds.shape[0]} vs rows={mask.sum()}"
            )

        # map 0/1/2 back to -1/0/1
        pred_flag_col[mask] = np.vectorize(inv_map.get)(preds)

    df_fe_valid["Demand_Response_Flag"] = pred_flag_col.astype(int)

    # 6) Write back into RAW dataframe
    df_out = df_raw.copy()
    df_out.loc[df_fe_valid["raw_row_id"].values, "Demand_Response_Flag"] = df_fe_valid["Demand_Response_Flag"].values

    # GUARANTEE: raw columns only
    df_out = df_out.loc[:, RAW_COLS]

    out_path = os.path.normpath(os.path.join(os.getcwd(), "../data/regression/regression-train-predflags.csv"))
    df_out.to_csv(out_path, index=False)
    print(f"Saved RAW-only CSV: {out_path}")


    keys = ["train/geometric_mean", "train/f1", "val/geometric_mean", "val/f1"]

    print("\nPer-fold:")
    for r in cv_results:
        print(r["fold"], {k: r[k] for k in keys})

    print("\nCV summary:")
    for k in keys:
        vals = np.array([r[k] for r in cv_results], dtype=float)
        mean = np.nanmean(vals)
        std  = np.nanstd(vals, ddof=1) if np.sum(~np.isnan(vals)) > 1 else np.nan
        print(f"{k}: mean={mean:.4f}, std={std:.4f}")


best_iteration,▁▁
best_score,▁
epoch,▁▁▁▁▂▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▆▆▇▇▇▇▇▇█
train-merror,█▆▆▅▇▄▄▅▅▅▄▄▄▄▄▄▄▄▄▃▃▃▃▃▃▃▃▂▂▂▂▁▁▁▁▂▂▂▂▂
train-mlogloss,███▇▇▇▇▆▆▆▆▆▅▅▅▅▄▄▄▄▄▃▃▃▃▃▃▃▂▂▂▂▂▂▂▁▁▁▁▁
train/accuracy,▁
train/f1,▁
train/geometric_mean,▁
val-merror,▃▁▅▆████▆▇██▇▇▇█▇▇▇▆▆▅▆▆▆▆▆▆▆▆▅▅▆▅▅▅▅▅▅▅
val-mlogloss,███▇▇▇▇▆▆▆▆▅▅▅▅▅▄▄▄▄▄▄▃▃▃▃▃▃▂▂▂▂▂▂▂▁▁▁▁▁
+3,...


best_iteration,▁▁
best_score,▁
epoch,▁▁▁▁▂▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▆▆▆▆▆▆▇▇▇▇▇▇███
train-merror,██▅▆▆▄▄▄▄▄▄▄▃▃▄▄▃▃▃▂▂▂▂▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁
train-mlogloss,███▇▇▇▇▆▆▆▅▅▅▅▅▅▄▄▄▄▄▃▃▃▃▃▃▃▂▂▂▂▂▂▂▁▁▁▁▁
train/accuracy,▁
train/f1,▁
train/geometric_mean,▁
val-merror,▁▇▃▄▃▄▆▆▇▄▇██▆▆▆▇▆▆▅▄▃▂▃▃▃▃▃▃▃▃▃▃▂▃▃▃▂▂▂
val-mlogloss,███▇▇▇▆▆▆▆▆▅▅▅▅▄▄▄▄▄▄▃▃▃▃▃▃▂▂▂▂▂▂▂▂▁▁▁▁▁
+3,...


best_iteration,▁▁
best_score,▁
epoch,▁▁▁▁▂▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▄▄▅▅▅▅▅▆▆▆▆▆▆▇▇▇▇▇██
train-merror,█▅▆▆▅▅▄▄▄▄▃▃▂▃▃▂▂▂▂▂▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁
train-mlogloss,██▇▇▇▆▆▆▆▆▅▅▅▅▅▄▄▄▄▄▄▃▃▃▃▃▃▃▂▂▂▂▂▂▂▂▁▁▁▁
train/accuracy,▁
train/f1,▁
train/geometric_mean,▁
val-merror,█▄▂▁▁▁▁▂▂▂▁▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▂
val-mlogloss,███▇▇▆▆▆▆▆▅▅▅▅▅▅▄▄▄▄▄▄▃▃▃▃▃▃▃▃▂▂▂▂▂▂▁▁▁▁
+3,...


C:\Users\timon\AppData\Local\Temp\ipykernel_21272\1060236595.py:98: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: g.iloc[ENTRIES_PER_DAY_LOCAL:])


Saved RAW-only CSV: c:\Users\timon\git\aicomp-flextrack\data\regression\regression-train-predflags.csv

Per-fold:
fold_1-val_SiteA {'train/geometric_mean': np.float64(0.8892880405723638), 'train/f1': np.float64(0.8503331530550394), 'val/geometric_mean': np.float64(0.7953504366719084), 'val/f1': np.float64(0.3616968063918154)}
fold_2-val_SiteB {'train/geometric_mean': np.float64(0.9024457965585431), 'train/f1': np.float64(0.8672886388338426), 'val/geometric_mean': np.float64(0.8010568133402922), 'val/f1': np.float64(0.4150947937119871)}
fold_3-val_SiteC {'train/geometric_mean': np.float64(0.9156651868386981), 'train/f1': np.float64(0.8844906729563616), 'val/geometric_mean': np.float64(0.7334560245256952), 'val/f1': np.float64(0.3963882373442398)}

CV summary:
train/geometric_mean: mean=0.9025, std=0.0132
train/f1: mean=0.8674, std=0.0171
val/geometric_mean: mean=0.7766, std=0.0375
val/f1: mean=0.3911, std=0.0271


In [29]:
import pandas as pd
import numpy as np
from sklearn.metrics import f1_score, confusion_matrix

# paths
true_path = "../data/regression/regression-train.csv"
pred_path = "../data/classification/classification-comp-prediction.csv"

# load
df_true = pd.read_csv(true_path)
df_pred = pd.read_csv(pred_path)

# align rows by key
key_cols = ["Site", "Timestamp_Local"]

df = (
    df_true[key_cols + ["Demand_Response_Flag"]]
    .rename(columns={"Demand_Response_Flag": "y_true"})
    .merge(
        df_pred[key_cols + ["Demand_Response_Flag"]]
        .rename(columns={"Demand_Response_Flag": "y_pred"}),
        on=key_cols,
        how="inner",
        validate="one_to_one",
    )
)

# drop rows where prediction is missing (e.g. first ENTRIES_PER_DAY)
df = df.dropna(subset=["y_pred"])

y_true = df["y_true"].astype(int).values
y_pred = df["y_pred"].astype(int).values

# -------------------
# F1 score (macro)
# -------------------
f1 = f1_score(y_true, y_pred, average="macro")

# -------------------
# Geometric Mean (G-Mean)
# -------------------
cm = confusion_matrix(y_true, y_pred, labels=[-1, 0, 1])

# recall per class
with np.errstate(divide="ignore", invalid="ignore"):
    recalls = np.diag(cm) / cm.sum(axis=1)

# remove classes with zero samples
recalls = recalls[~np.isnan(recalls)]

gmean = np.prod(recalls) ** (1 / len(recalls))

# -------------------
print(f"Samples evaluated: {len(y_true)}")
print(f"Macro F1 score:    {f1:.4f}")
print(f"Geometric Mean:    {gmean:.4f}")


Samples evaluated: 105120
Macro F1 score:    0.4202
Geometric Mean:    0.7343
